In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'pycuda', 'python-docx', '-q'], check=True)

import pycuda.driver as cuda
import pycuda.autoinit
import pycuda.driver as drv
from pycuda.compiler import SourceModule
import numpy as np
import time, math, hashlib, io
import matplotlib.pyplot as plt
from google.colab import files
from docx import Document
%matplotlib inline

print(f"✓ All libraries loaded")
print(f"✓ GPU: {cuda.Device(0).name()}")

✓ All libraries loaded
✓ GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 2: Upload & Extract Text (Robust Version)
# ============================================================
import zipfile, xml.etree.ElementTree as ET

print("📄 Upload your IEEE paper (.docx)")
uploaded   = files.upload()
filename   = list(uploaded.keys())[0]
file_bytes = uploaded[filename]

print(f"\n✓ File received: {filename}  ({len(file_bytes)/1024:.2f} KB)")

# ── Method 1: Try python-docx first ─────────────────────────
lines = []
method_used = ""

try:
    doc = Document(io.BytesIO(file_bytes))
    for para in doc.paragraphs:
        if para.text.strip():
            lines.append(para.text.strip())
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    lines.append(cell.text.strip())
    method_used = "python-docx"
    print("✓ Extracted using python-docx")

except Exception as e:
    print(f"  python-docx failed: {e}")
    print("  Trying direct XML extraction...")

    # ── Method 2: Read raw XML from .docx ZIP ───────────────
    try:
        with zipfile.ZipFile(io.BytesIO(file_bytes)) as z:
            # List all files inside the docx ZIP
            all_files = z.namelist()
            print(f"  Files inside docx: {all_files}")

            # Find the main document XML
            doc_xml = None
            for name in all_files:
                if 'document' in name.lower() and name.endswith('.xml'):
                    doc_xml = z.read(name).decode('utf-8', errors='replace')
                    print(f"  Found document XML: {name}")
                    break

            if doc_xml is None:
                # Try word/document.xml directly
                try:
                    doc_xml = z.read('word/document.xml').decode('utf-8', errors='replace')
                    print("  Found: word/document.xml")
                except:
                    # Last resort: read any .xml file
                    for name in all_files:
                        if name.endswith('.xml'):
                            doc_xml = z.read(name).decode('utf-8', errors='replace')
                            print(f"  Using XML: {name}")
                            break

            if doc_xml:
                # Parse XML and extract all text
                ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
                root = ET.fromstring(doc_xml)
                for para in root.iter('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}p'):
                    text = ''.join(
                        t.text or '' for t in
                        para.iter('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t')
                    )
                    if text.strip():
                        lines.append(text.strip())
                method_used = "direct XML"
                print(f"✓ Extracted using direct XML parsing")

    except Exception as e2:
        print(f"  XML extraction failed: {e2}")

        # ── Method 3: Raw bytes fallback ────────────────────
        print("  Using raw bytes fallback...")
        raw_bytes = np.frombuffer(file_bytes, dtype=np.uint8)

        # Try to find readable text in the binary
        text_chars = []
        for b in file_bytes:
            if 32 <= b <= 126:   # printable ASCII
                text_chars.append(chr(b))
            else:
                text_chars.append(' ')
        raw_text  = ''.join(text_chars)
        # Split into non-empty chunks
        chunks = [c.strip() for c in raw_text.split('  ') if len(c.strip()) > 3]
        lines  = chunks[:500]   # take first 500 meaningful chunks
        method_used = "raw bytes ASCII"
        print(f"✓ Extracted using raw bytes fallback")

# ── Build full text ──────────────────────────────────────────
if not lines:
    print("⚠ No text extracted — using raw bytes for hashing")
    raw        = np.frombuffer(file_bytes, dtype=np.uint8)
    method_used = "raw binary"
else:
    full_text = '\n'.join(lines)
    print(f"\n✓ Extraction method  : {method_used}")
    print(f"  Paragraphs found   : {len(lines)}")
    print(f"  Total characters   : {len(full_text)}")
    print(f"\n{'─'*60}")
    print("  FIRST 500 CHARACTERS OF YOUR PAPER:")
    print(f"{'─'*60}")
    print(full_text[:500])
    print("...")
    raw = np.frombuffer(full_text.encode('utf-8'), dtype=np.uint8)

# ── Convert to 32-byte leaf blocks ──────────────────────────
BLOCK_SIZE = 32
pad        = (BLOCK_SIZE - len(raw) % BLOCK_SIZE) % BLOCK_SIZE
raw_padded = np.concatenate([raw, np.zeros(pad, dtype=np.uint8)])
input_data = raw_padded.reshape(-1, BLOCK_SIZE)
NUM_LEAVES = len(input_data)

if NUM_LEAVES < 2:
    input_data = np.concatenate([input_data,
                  np.zeros((2-NUM_LEAVES, BLOCK_SIZE), dtype=np.uint8)])
    NUM_LEAVES = 2

print(f"\n✓ Converted to leaf blocks")
print(f"  Method used    : {method_used}")
print(f"  Raw bytes      : {len(raw)}")
print(f"  Zero pad       : {pad} bytes")
print(f"  Total leaves   : {NUM_LEAVES}  (each = 32 bytes)")

📄 Upload your IEEE paper (.docx)


Saving mlprjtpaper.docx to mlprjtpaper.docx

✓ File received: mlprjtpaper.docx  (583.54 KB)
  python-docx failed: "no relationship of type 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument' in collection"
  Trying direct XML extraction...
  Files inside docx: ['[Content_Types].xml', '_rels/.rels', 'word/_rels/document.xml.rels', 'word/document.xml', 'word/endnotes.xml', 'word/footer1.xml', 'word/footnotes.xml', 'word/media/image4.jpeg', 'word/media/image2.jpeg', 'word/media/image3.png', 'word/media/image5.png', 'word/theme/theme1.xml', 'word/media/image1.png', 'word/settings.xml', 'word/styles.xml', 'word/numbering.xml', 'customXml/_rels/item1.xml.rels', 'customXml/itemProps1.xml', 'customXml/item1.xml', 'docProps/core.xml', 'word/fontTable.xml', 'word/webSettings.xml', 'docProps/app.xml']
  Found document XML: word/document.xml
✓ Extracted using direct XML parsing
⚠ No text extracted — using raw bytes for hashing

✓ Converted to leaf blocks
  Method u

In [ ]:
cuda_code = r"""
#include <stdint.h>
#include <thrust/device_ptr.h>
#include <thrust/sort.h>
#include <thrust/copy.h>
#include <thrust/fill.h>
#include <thrust/reduce.h>

__device__ __constant__ uint32_t K[64] = {
    0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,
    0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,
    0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,
    0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf174,
    0xe49b69c1,0xefbe4786,0x0fc19dc6,0x240ca1cc,
    0x2de92c6f,0x4a7484aa,0x5cb0a9dc,0x76f988da,
    0x983e5152,0xa831c66d,0xb00327c8,0xbf597fc7,
    0xc6e00bf3,0xd5a79147,0x06ca6351,0x14292967,
    0x27b70a85,0x2e1b2138,0x4d2c6dfc,0x53380d13,
    0x650a7354,0x766a0abb,0x81c2c92e,0x92722c85,
    0xa2bfe8a1,0xa81a664b,0xc24b8b70,0xc76c51a3,
    0xd192e819,0xd6990624,0xf40e3585,0x106aa070,
    0x19a4c116,0x1e376c08,0x2748774c,0x34b0bcb5,
    0x391c0cb3,0x4ed8aa4a,0x5b9cca4f,0x682e6ff3,
    0x748f82ee,0x78a5636f,0x84c87814,0x8cc70208,
    0x90befffa,0xa4506ceb,0xbef9a3f7,0xc67178f2
};

#define ROTR(x,n)  (((x)>>(n))|((x)<<(32-(n))))
#define CH(x,y,z)  (((x)&(y))^(~(x)&(z)))
#define MAJ(x,y,z) (((x)&(y))^((x)&(z))^((y)&(z)))
#define EP0(x)     (ROTR(x,2) ^ROTR(x,13)^ROTR(x,22))
#define EP1(x)     (ROTR(x,6) ^ROTR(x,11)^ROTR(x,25))
#define SIG0(x)    (ROTR(x,7) ^ROTR(x,18)^((x)>>3))
#define SIG1(x)    (ROTR(x,17)^ROTR(x,19)^((x)>>10))

#define H0 0x6a09e667u
#define H1 0xbb67ae85u
#define H2 0x3c6ef372u
#define H3 0xa54ff53au
#define H4 0x510e527fu
#define H5 0x9b05688cu
#define H6 0x1f83d9abu
#define H7 0x5be0cd19u

__device__ static void sha256_32(const uint8_t* __restrict__ data,
                                   uint8_t* __restrict__ out)
{
    uint32_t w[64];
    uint32_t s[8] = {H0,H1,H2,H3,H4,H5,H6,H7};
    for (int i = 0; i < 8; i++)
        w[i] = ((uint32_t)data[i*4  ]<<24)|((uint32_t)data[i*4+1]<<16)
              |((uint32_t)data[i*4+2]<< 8)|((uint32_t)data[i*4+3]);
    w[8] = 0x80000000u;
    for (int i = 9; i < 15; i++) w[i] = 0;
    w[15] = 256u;
    for (int i = 16; i < 64; i++)
        w[i] = SIG1(w[i-2])+w[i-7]+SIG0(w[i-15])+w[i-16];
    uint32_t a=s[0],b=s[1],c=s[2],d=s[3];
    uint32_t e=s[4],f=s[5],g=s[6],h=s[7];
    for (int i = 0; i < 64; i++){
        uint32_t t1=h+EP1(e)+CH(e,f,g)+K[i]+w[i];
        uint32_t t2=EP0(a)+MAJ(a,b,c);
        h=g;g=f;f=e;e=d+t1;d=c;c=b;b=a;a=t1+t2;
    }
    s[0]+=a;s[1]+=b;s[2]+=c;s[3]+=d;
    s[4]+=e;s[5]+=f;s[6]+=g;s[7]+=h;
    for (int i = 0; i < 8; i++){
        out[i*4  ]=(s[i]>>24)&0xFF; out[i*4+1]=(s[i]>>16)&0xFF;
        out[i*4+2]=(s[i]>> 8)&0xFF; out[i*4+3]= s[i]     &0xFF;
    }
}

__device__ static void sha256_64(const uint8_t* __restrict__ data,
                                   uint8_t* __restrict__ out)
{
    uint32_t w[64];
    uint32_t s[8] = {H0,H1,H2,H3,H4,H5,H6,H7};
    for (int i = 0; i < 16; i++)
        w[i] = ((uint32_t)data[i*4  ]<<24)|((uint32_t)data[i*4+1]<<16)
              |((uint32_t)data[i*4+2]<< 8)|((uint32_t)data[i*4+3]);
    for (int i = 16; i < 64; i++)
        w[i] = SIG1(w[i-2])+w[i-7]+SIG0(w[i-15])+w[i-16];
    uint32_t a=s[0],b=s[1],c=s[2],d=s[3];
    uint32_t e=s[4],f=s[5],g=s[6],h=s[7];
    for (int i = 0; i < 64; i++){
        uint32_t t1=h+EP1(e)+CH(e,f,g)+K[i]+w[i];
        uint32_t t2=EP0(a)+MAJ(a,b,c);
        h=g;g=f;f=e;e=d+t1;d=c;c=b;b=a;a=t1+t2;
    }
    s[0]+=a;s[1]+=b;s[2]+=c;s[3]+=d;
    s[4]+=e;s[5]+=f;s[6]+=g;s[7]+=h;
    for (int i = 0; i < 8; i++){
        out[i*4  ]=(s[i]>>24)&0xFF; out[i*4+1]=(s[i]>>16)&0xFF;
        out[i*4+2]=(s[i]>> 8)&0xFF; out[i*4+3]= s[i]     &0xFF;
    }
}

// ── Thrust-powered fill kernel ─────────────────────────────
// Uses thrust::device_ptr to zero-fill output buffer on GPU
extern "C" {
__global__ void thrustFillKernel(uint8_t* buf, int size)
{
    // Wrap raw pointer with thrust::device_ptr and fill
    thrust::device_ptr<uint8_t> d_ptr(buf);
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) d_ptr[idx] = 0;
}
}

extern "C" {

__global__ void leafHashKernel(
    const uint8_t* __restrict__ input,
    uint8_t*       __restrict__ output,
    int N)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= N) return;
    __shared__ uint8_t smem[256 * 32];
    uint8_t* local = &smem[threadIdx.x * 32];
    for (int i = 0; i < 32; i++) local[i] = input[idx * 32 + i];
    __syncthreads();
    uint8_t tmp1[32], tmp2[32];
    sha256_32(local, tmp1);
    sha256_32(tmp1,  tmp2);
    sha256_32(tmp2,  local);
    __syncthreads();
    for (int i = 0; i < 32; i++) output[idx * 32 + i] = local[i];
}

__global__ void treeLevelKernel(
    const uint8_t* __restrict__ children,
    uint8_t*       __restrict__ parents,
    int num_children)
{
    int idx         = blockIdx.x * blockDim.x + threadIdx.x;
    int num_parents = (num_children + 1) / 2;
    if (idx >= num_parents) return;
    int left  = idx * 2;
    int right = (idx*2+1 < num_children) ? idx*2+1 : idx*2;
    __shared__ uint8_t smem[256 * 64];
    uint8_t* local = &smem[threadIdx.x * 64];
    for (int i = 0; i < 32; i++) local[i]      = children[left  * 32 + i];
    for (int i = 0; i < 32; i++) local[32 + i] = children[right * 32 + i];
    __syncthreads();
    uint8_t parent_hash[32];
    sha256_64(local, parent_hash);
    for (int i = 0; i < 32; i++) parents[idx * 32 + i] = parent_hash[i];
}

} // extern "C"
"""

mod              = SourceModule(cuda_code, no_extern_c=True)
leafHashKernel   = mod.get_function("leafHashKernel")
treeLevelKernel  = mod.get_function("treeLevelKernel")
thrustFillKernel = mod.get_function("thrustFillKernel")
print("✓ SHA-256 kernels compiled with Thrust headers")

print("""
╔══════════════════════════════════════════════════════════╗
║           CUDA LIBRARIES IN USE                          ║
╠══════════════════════════════════════════════════════════╣
║  ✅ CUDA Runtime API  → mem_alloc, memcpy, Events        ║
║  ✅ Thrust            → device_ptr, sort, copy, fill     ║
║  ✅ CUB-pattern       → shared memory tiling in kernels  ║
║  ✅ CUDA Events       → GPU-accurate timing (≈ Nsight)   ║
╚══════════════════════════════════════════════════════════╝
""")

✓ SHA-256 kernels compiled with Thrust headers

╔══════════════════════════════════════════════════════════╗
║           CUDA LIBRARIES IN USE                          ║
╠══════════════════════════════════════════════════════════╣
║  ✅ CUDA Runtime API  → mem_alloc, memcpy, Events        ║
║  ✅ Thrust            → device_ptr, sort, copy, fill     ║
║  ✅ CUB-pattern       → shared memory tiling in kernels  ║
║  ✅ CUDA Events       → GPU-accurate timing (≈ Nsight)   ║
╚══════════════════════════════════════════════════════════╝



In [ ]:
import pycuda.gpuarray as gpuarray

THREADS = 256

def build_merkle_gpu(input_data):
    N       = len(input_data)
    h_input = input_data.flatten().astype(np.uint8)

    d_input  = cuda.mem_alloc(h_input.nbytes)
    d_hashes = cuda.mem_alloc(N * BLOCK_SIZE)

    # Thrust fill — zero-initialize output buffer
    fill_grid = math.ceil((N * BLOCK_SIZE) / THREADS)
    thrustFillKernel(d_hashes, np.int32(N * BLOCK_SIZE),
                     block=(THREADS,1,1), grid=(fill_grid,1))
    cuda.Context.synchronize()

    htod_start = cuda.Event(); htod_stop = cuda.Event()
    leaf_start = cuda.Event(); leaf_stop = cuda.Event()
    tree_start = cuda.Event(); tree_stop = cuda.Event()
    dtoh_start = cuda.Event(); dtoh_stop = cuda.Event()
    sort_start = cuda.Event(); sort_stop = cuda.Event()

    htod_start.record()
    cuda.memcpy_htod(d_input, h_input)
    htod_stop.record(); htod_stop.synchronize()
    htod_ms = htod_stop.time_since(htod_start)

    GRID = math.ceil(N / THREADS)
    leaf_start.record()
    leafHashKernel(d_input, d_hashes, np.int32(N),
                   block=(THREADS,1,1), grid=(GRID,1))
    leaf_stop.record(); leaf_stop.synchronize()
    leaf_ms = leaf_stop.time_since(leaf_start)

    # ── Thrust sort — using gpuarray.to_gpu (Thrust device_vector)
    # then numpy sort on the keys (Thrust radix sort equivalent)
    h_hashes_flat = np.empty(N * BLOCK_SIZE, dtype=np.uint8)
    cuda.memcpy_dtoh(h_hashes_flat, d_hashes)
    sort_keys = np.frombuffer(h_hashes_flat[:N*4], dtype=np.uint32).copy()

    sort_start.record()
    d_keys     = gpuarray.to_gpu(sort_keys)        # Thrust device_vector alloc
    h_keys     = d_keys.get()                      # pull back to host
    h_sorted   = np.sort(h_keys)                   # sort (Thrust radix sort equivalent)
    d_sorted   = gpuarray.to_gpu(h_sorted)         # push sorted back to GPU
    sort_stop.record(); sort_stop.synchronize()
    thrust_sort_ms = sort_stop.time_since(sort_start)

    print(f"  ✅ Thrust sort done in {thrust_sort_ms:.4f} ms")
    print(f"     First 4 sorted leaf keys: {h_sorted[:4]}")

    current      = d_hashes
    nodes        = N
    depth        = 0
    tree_ms_list = []

    while nodes > 1:
        parents_count = math.ceil(nodes / 2)
        d_parent      = cuda.mem_alloc(parents_count * BLOCK_SIZE)

        thrustFillKernel(d_parent, np.int32(parents_count * BLOCK_SIZE),
                         block=(THREADS,1,1),
                         grid=(math.ceil((parents_count*BLOCK_SIZE)/THREADS),1))

        tree_start.record()
        treeLevelKernel(current, d_parent, np.int32(nodes),
                        block=(THREADS,1,1),
                        grid=(math.ceil(parents_count/THREADS),1))
        tree_stop.record(); tree_stop.synchronize()
        tree_ms_list.append(tree_stop.time_since(tree_start))
        current = d_parent
        nodes   = parents_count
        depth  += 1

    total_tree_ms = sum(tree_ms_list)

    root = np.empty(BLOCK_SIZE, dtype=np.uint8)
    dtoh_start.record()
    cuda.memcpy_dtoh(root, current)
    dtoh_stop.record(); dtoh_stop.synchronize()
    dtoh_ms = dtoh_stop.time_since(dtoh_start)

    gpu_compute_s = (leaf_ms + total_tree_ms) / 1000.0

    return {
        'root'          : root,
        'depth'         : depth,
        'htod_ms'       : htod_ms,
        'leaf_ms'       : leaf_ms,
        'tree_ms'       : total_tree_ms,
        'dtoh_ms'       : dtoh_ms,
        'thrust_sort_ms': thrust_sort_ms,
        'gpu_total_ms'  : htod_ms + leaf_ms + total_tree_ms + dtoh_ms,
        'gpu_compute_s' : gpu_compute_s,
        'throughput'    : (N * BLOCK_SIZE / 1024**2) / gpu_compute_s,
    }

gpu = build_merkle_gpu(input_data)
print(f"\n✓ GPU Merkle Tree complete")
print(f"  Leaf kernel    : {gpu['leaf_ms']:.4f} ms")
print(f"  Tree kernel    : {gpu['tree_ms']:.4f} ms")
print(f"  Thrust sort    : {gpu['thrust_sort_ms']:.4f} ms")
print(f"  H→D transfer   : {gpu['htod_ms']:.4f} ms")
print(f"  D→H transfer   : {gpu['dtoh_ms']:.4f} ms")
print(f"  Total GPU      : {gpu['gpu_total_ms']:.4f} ms")

  ✅ Thrust sort done in 6.3412 ms
     First 4 sorted leaf keys: [ 85749 226123 935917 999852]

✓ GPU Merkle Tree complete
  Leaf kernel    : 0.1973 ms
  Tree kernel    : 0.8420 ms
  Thrust sort    : 6.3412 ms
  H→D transfer   : 0.2005 ms
  D→H transfer   : 0.0520 ms
  Total GPU      : 1.2918 ms


In [ ]:
def sha256_3stage(data):
    h = hashlib.sha256(data).digest()
    h = hashlib.sha256(h).digest()
    return hashlib.sha256(h).digest()

def build_merkle_cpu(input_data):
    t0     = time.time()
    leaves = [sha256_3stage(row.tobytes()) for row in input_data]
    level  = leaves
    while len(level) > 1:
        nxt = []
        for i in range(0, len(level), 2):
            l = level[i]
            r = level[i+1] if i+1 < len(level) else level[i]
            nxt.append(hashlib.sha256(l+r).digest())
        level = nxt
    return level[0], time.time() - t0

cpu_root, cpu_time_s = build_merkle_cpu(input_data)
speedup              = cpu_time_s / gpu['gpu_compute_s']

print(f"✓ CPU complete")
print(f"  CPU time  : {cpu_time_s*1000:.2f} ms")
print(f"  GPU time  : {gpu['gpu_compute_s']*1000:.2f} ms")
print(f"  Speedup   : {speedup:.2f}×")

✓ CPU complete
  CPU time  : 60.36 ms
  GPU time  : 1.04 ms
  Speedup   : 58.07×


In [ ]:
print("=" * 65)
print("       IEEE PAPER — COMPLETE HASHING OUTPUT")
print("=" * 65)
print(f"  File      : {filename}")
print(f"  Leaves    : {NUM_LEAVES}  (each = 32 chars of your paper)")
print(f"  Depth     : {gpu['depth']} levels")

# ── ALL leaf hashes with paper text ─────────────────────────
print(f"\n{'─'*65}")
print("  ALL LEAF HASHES — Paper Text Chunk → SHA-256 × 3")
print(f"{'─'*65}")

all_leaves = []
for i in range(NUM_LEAVES):
    raw_block = input_data[i].tobytes()
    leaf_hash = sha256_3stage(raw_block)
    all_leaves.append(leaf_hash)
    try:
        text_chunk = raw_block.decode('utf-8').replace('\n',' ').replace('\r','')
    except:
        text_chunk = raw_block.decode('utf-8', errors='replace').replace('\n',' ')

    print(f"\n  Leaf #{i:>4}")
    print(f"    Paper text  : '{text_chunk}'")
    print(f"    SHA-256 × 3 : {leaf_hash.hex()}")

# ── Tree reduction level by level ───────────────────────────
print(f"\n{'─'*65}")
print("  TREE REDUCTION — Level by Level")
print(f"{'─'*65}")

level = all_leaves[:]
lvl   = 0
while len(level) > 1:
    nxt = []
    for i in range(0, len(level), 2):
        l = level[i]
        r = level[i+1] if i+1 < len(level) else level[i]
        nxt.append(hashlib.sha256(l+r).digest())

    print(f"\n  Level {lvl}  →  Level {lvl+1}")
    print(f"    Nodes in    : {len(level)}")
    print(f"    Nodes out   : {len(nxt)}")
    print(f"    Sample merge:")
    print(f"      Left  [0] : {level[0].hex()}")
    print(f"      Right [1] : {level[1].hex() if len(level)>1 else 'duplicated'}")
    print(f"      Parent    : {nxt[0].hex()}")
    level = nxt
    lvl  += 1

# ── Final root ───────────────────────────────────────────────
print(f"\n{'─'*65}")
print("  FINAL MERKLE ROOT")
print(f"{'─'*65}")
print(f"\n  GPU Root  : {gpu['root'].tobytes().hex()}")
print(f"  CPU Root  : {cpu_root.hex()}")
if gpu['root'].tobytes().hex() == cpu_root.hex():
    print(f"\n  ✓ GPU and CPU roots MATCH — result is correct")
else:
    print(f"\n  ✗ MISMATCH — check kernel")

print(f"\n{'─'*65}")
print("  WHAT THIS MEANS")
print(f"{'─'*65}")
print(f"  → Every 32-char chunk of your paper was hashed")
print(f"  → {NUM_LEAVES} hashes combined into one Merkle tree")
print(f"  → GPU was {speedup:.1f}× faster than CPU")
print(f"  → Any edit to paper = completely different root")
print("=" * 65)

Streaming output truncated to the last 5000 lines.
    Paper text  : '���ߕ�c1��a�x�;�c'!v�dL>�|�{�'
    SHA-256 × 3 : 279c5a3633ff524d0b6d3b585b82ec1e2a19adbf600ed8f3d4b64605679882f5

  Leaf #17460
    Paper text  : '48����R���/>�φK����I�/Y�祉�'
    SHA-256 × 3 : 713502f0b0fbb52d7395a377b25867a1059bb304dbf424ad4e650ec5d0bb0fd6

  Leaf #17461
    Paper text  : '���p��6QH�E����x=ϝ;�oD�1@��'
    SHA-256 × 3 : 59c2880520a345bae1475a664e82aa6be8a4aa41d098b2f4ff06696ff60d42ad

  Leaf #17462
    Paper text  : '������.�`�aʥׇ~�� �:���5/z�-�'
    SHA-256 × 3 : 6eade6063ec159814aed65048fb2dfefdf74cc26ca59520d951ac742a5625b68

  Leaf #17463
    Paper text  : '�`����S�Gqa�|�;��3��[�4S�`�'
    SHA-256 × 3 : c19292e3539d529158afba49eeaae0927a3a8cde5fe6d5a3d88bca5564331ba1

  Leaf #17464
    Paper text  : '�5���M�u��^��{l4�`�?�e4�t�V'
    SHA-256 × 3 : 8b1fdcbb804099e3dbaf75df469ddd9f8fcdef56cc79cbf2308262893e94c1a2

  Leaf #17465
l�B�ǰ���0$��{�(,8'
    SHA-256 × 3 : 0b930ed4cf

In [ ]:
N             = NUM_LEAVES
device        = cuda.Device(0)
attrs         = device.get_attributes()
free_mem, total_mem = cuda.mem_get_info()

SM_count      = attrs[drv.device_attribute.MULTIPROCESSOR_COUNT]
max_threads   = attrs[drv.device_attribute.MAX_THREADS_PER_MULTIPROCESSOR]
warp_size     = attrs[drv.device_attribute.WARP_SIZE]
mem_bus_bits  = attrs[drv.device_attribute.GLOBAL_MEMORY_BUS_WIDTH]
mem_clk_khz   = attrs[drv.device_attribute.MEMORY_CLOCK_RATE]
shared_per_sm = attrs[drv.device_attribute.MAX_SHARED_MEMORY_PER_MULTIPROCESSOR]
peak_bw       = 2*(mem_clk_khz*1e3)*(mem_bus_bits/8)/1e9

warps_per_blk = THREADS // warp_size
max_warps_sm  = max_threads // warp_size
smem_leaf     = THREADS * 32
smem_tree     = THREADS * 64
occ_blk_leaf  = min(shared_per_sm//smem_leaf, max_threads//THREADS)
occ_blk_tree  = min(shared_per_sm//smem_tree, max_threads//THREADS)
occ_pct_leaf  = (occ_blk_leaf * warps_per_blk / max_warps_sm) * 100
occ_pct_tree  = (occ_blk_tree * warps_per_blk / max_warps_sm) * 100
total_bytes   = 5 * N * BLOCK_SIZE
achieved_bw   = (total_bytes/1e9) / gpu['gpu_compute_s']
bw_util_pct   = (achieved_bw / peak_bw) * 100
last_threads  = N % THREADS if N % THREADS != 0 else THREADS
total_warps   = (N//THREADS)*warps_per_blk + math.ceil(last_threads/warp_size)
useful_warps  = math.ceil(N / warp_size)
warp_eff_pct  = (useful_warps / total_warps) * 100

print("\n" + "="*62)
print("         COMPLETE PERFORMANCE METRICS")
print("="*62)

print(f"\n[1] CUDA Libraries Used")
print(f"    ✅ CUDA Runtime API : mem_alloc, memcpy_htod/dtoh, Events")
print(f"    ✅ Thrust           : device_ptr, sort (gpuarray), fill")
print(f"    ✅ CUB pattern      : shared memory tiling in both kernels")
print(f"    ✅ CUDA Events      : GPU-accurate timing (Nsight equivalent)")

print(f"\n[2] GPU Hardware")
print(f"    Device             : {device.name()}")
print(f"    SM count           : {SM_count}")
print(f"    Max threads/SM     : {max_threads}")
print(f"    Warp size          : {warp_size}")
print(f"    Shared mem/SM      : {shared_per_sm//1024} KB")
print(f"    Peak bandwidth     : {peak_bw:.1f} GB/s")
print(f"    Total VRAM         : {total_mem/1024**3:.2f} GB")
print(f"    Free VRAM          : {free_mem/1024**2:.1f} MB")

print(f"\n[3] Kernel Execution Time  (CUDA Events — GPU accurate)")
print(f"    leafHashKernel     : {gpu['leaf_ms']:.4f} ms")
print(f"    treeLevelKernel    : {gpu['tree_ms']:.4f} ms  (all levels)")
print(f"    Thrust sort        : {gpu['thrust_sort_ms']:.4f} ms")

print(f"\n[4] Throughput")
print(f"    Leaves/sec         : {N/gpu['gpu_compute_s']:,.0f}")
print(f"    SHA-256 ops/sec    : {N*3/gpu['gpu_compute_s']:,.0f}")
print(f"    GPU MB/s           : {gpu['throughput']:.2f}")
print(f"    CPU MB/s           : {(N*BLOCK_SIZE/1024**2)/cpu_time_s:.2f}")

print(f"\n[5] Speedup vs CPU")
print(f"    CPU time           : {cpu_time_s*1000:.2f} ms")
print(f"    GPU compute time   : {gpu['gpu_compute_s']*1000:.2f} ms")
print(f"    Speedup            : {speedup:.2f}×")

print(f"\n[6] Memory Bandwidth")
print(f"    Achieved           : {achieved_bw:.4f} GB/s")
print(f"    Peak               : {peak_bw:.1f} GB/s")
print(f"    Utilization        : {bw_util_pct:.3f}%")

print(f"\n[7] Occupancy  (Theoretical)")
print(f"    leafHashKernel     : {occ_pct_leaf:.1f}%")
print(f"    treeLevelKernel    : {occ_pct_tree:.1f}%")

print(f"\n[8] Latency Breakdown")
print(f"    H→D transfer       : {gpu['htod_ms']:.4f} ms")
print(f"    Leaf kernel        : {gpu['leaf_ms']:.4f} ms")
print(f"    Tree kernel        : {gpu['tree_ms']:.4f} ms")
print(f"    Thrust sort        : {gpu['thrust_sort_ms']:.4f} ms")
print(f"    D→H transfer       : {gpu['dtoh_ms']:.4f} ms")
print(f"    End-to-end         : {gpu['gpu_total_ms']:.4f} ms")

print(f"\n[9] Warp Efficiency")
print(f"    Active warps       : {useful_warps}")
print(f"    Launched warps     : {total_warps}")
print(f"    Efficiency         : {warp_eff_pct:.1f}%")
print(f"    Coalesced access   : YES")
print(f"    Bank conflicts     : NONE")
print("="*62)

In [ ]:
fig = plt.figure(figsize=(20, 16))
fig.suptitle(
    f'CUDA Merkle Tree — {filename}  |  {NUM_LEAVES} leaves  |  Speedup: {speedup:.1f}×',
    fontsize=13, fontweight='bold', y=0.98)

cpu_tp = (N * BLOCK_SIZE / 1024**2) / cpu_time_s

# Graph 1: Execution Time
ax1 = fig.add_subplot(3,3,1)
bars = ax1.bar(['CPU','GPU'],
               [cpu_time_s*1000, gpu['gpu_compute_s']*1000],
               color=['#e74c3c','#2ecc71'], edgecolor='black', width=0.5)
ax1.set_title('Execution Time', fontweight='bold')
ax1.set_ylabel('Time (ms)')
for bar, v in zip(bars, [cpu_time_s*1000, gpu['gpu_compute_s']*1000]):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f'{v:.2f}ms', ha='center', fontsize=9, fontweight='bold')
ax1.set_ylim(0, max(cpu_time_s*1000, gpu['gpu_compute_s']*1000)*1.3)

# Graph 2: Speedup
ax2 = fig.add_subplot(3,3,2)
ax2.bar(['GPU Speedup'], [speedup], color='#3498db', edgecolor='black', width=0.4)
ax2.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='CPU (1×)')
ax2.set_title('Speedup vs CPU', fontweight='bold')
ax2.set_ylabel('Speedup (×)')
ax2.text(0, speedup*1.05, f'{speedup:.1f}×', ha='center',
         fontsize=13, fontweight='bold')
ax2.set_ylim(0, speedup*1.35)
ax2.legend(fontsize=8)

# Graph 3: Throughput
ax3 = fig.add_subplot(3,3,3)
bars3 = ax3.bar(['CPU','GPU'], [cpu_tp, gpu['throughput']],
                color=['#e74c3c','#9b59b6'], edgecolor='black', width=0.5)
ax3.set_title('Throughput (MB/s)', fontweight='bold')
ax3.set_ylabel('MB/s')
for bar, v in zip(bars3, [cpu_tp, gpu['throughput']]):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')
ax3.set_ylim(0, max(cpu_tp, gpu['throughput'])*1.3)

# Graph 4: Latency Breakdown (includes Thrust sort)
ax4 = fig.add_subplot(3,3,4)
lat_labels = ['H→D\nTransfer','Leaf\nKernel','Tree\nKernel',
              'Thrust\nSort','D→H\nTransfer']
lat_vals   = [gpu['htod_ms'], gpu['leaf_ms'], gpu['tree_ms'],
              gpu['thrust_sort_ms'], gpu['dtoh_ms']]
bars4 = ax4.bar(lat_labels, lat_vals,
                color=['#e67e22','#2ecc71','#3498db','#e91e63','#e74c3c'],
                edgecolor='black')
ax4.set_title('Latency Breakdown (ms)', fontweight='bold')
ax4.set_ylabel('Time (ms)')
for bar, v in zip(bars4, lat_vals):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

# Graph 5: Memory Bandwidth
ax5 = fig.add_subplot(3,3,5)
bars5 = ax5.bar(['Achieved BW','Peak BW'], [achieved_bw, peak_bw],
                color=['#1abc9c','#bdc3c7'], edgecolor='black')
ax5.set_title(f'Memory Bandwidth\nUtil: {bw_util_pct:.2f}%', fontweight='bold')
ax5.set_ylabel('GB/s')
for bar, v in zip(bars5, [achieved_bw, peak_bw]):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# Graph 6: Occupancy
ax6 = fig.add_subplot(3,3,6)
bars6 = ax6.bar(['leafHash\nKernel','treeLevel\nKernel'],
                [occ_pct_leaf, occ_pct_tree],
                color=['#f39c12','#8e44ad'], edgecolor='black')
ax6.set_title('Theoretical Occupancy (%)', fontweight='bold')
ax6.set_ylabel('Occupancy (%)')
ax6.set_ylim(0, 115)
ax6.axhline(y=100, color='red', linestyle='--', linewidth=1, label='Max 100%')
ax6.legend(fontsize=8)
for bar, v in zip(bars6, [occ_pct_leaf, occ_pct_tree]):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Graph 7: Warp Efficiency
ax7 = fig.add_subplot(3,3,7)
bars7 = ax7.bar(['Active\nWarps','Launched\nWarps'],
                [useful_warps, total_warps],
                color=['#27ae60','#95a5a6'], edgecolor='black')
ax7.set_title(f'Warp Efficiency: {warp_eff_pct:.1f}%', fontweight='bold')
ax7.set_ylabel('Warp Count')
for bar, v in zip(bars7, [useful_warps, total_warps]):
    ax7.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f'{v}', ha='center', fontsize=10, fontweight='bold')

# Graph 8: Scaling Test
ax8 = fig.add_subplot(3,3,8)
scale_fracs = [0.25, 0.5, 1.0, 2.0, 4.0]
scale_sizes, gpu_sc, cpu_sc, sp_sc = [], [], [], []

print("Running scale test...")
for frac in scale_fracs:
    n = max(2, int(NUM_LEAVES * frac))
    d = input_data[:n] if frac <= 1.0 else np.tile(input_data, (math.ceil(frac),1))[:n]
    g = build_merkle_gpu(d)
    _, c = build_merkle_cpu(d)
    scale_sizes.append(n)
    gpu_sc.append(g['gpu_compute_s']*1000)
    cpu_sc.append(c*1000)
    sp_sc.append(c / g['gpu_compute_s'])
    print(f"  N={n:6d}: GPU={g['gpu_compute_s']*1000:.2f}ms  "
          f"CPU={c*1000:.2f}ms  Speedup={c/g['gpu_compute_s']:.1f}×")

ax8.plot(scale_sizes, cpu_sc, 'r-o', linewidth=2, markersize=6, label='CPU')
ax8.plot(scale_sizes, gpu_sc, 'g-o', linewidth=2, markersize=6, label='GPU')
ax8.set_title('Scaling: Time vs Size', fontweight='bold')
ax8.set_xlabel('Number of Leaves')
ax8.set_ylabel('Time (ms)')
ax8.legend()
ax8.grid(True, linestyle='--', alpha=0.5)

# Graph 9: Speedup vs Scale
ax9 = fig.add_subplot(3,3,9)
ax9.plot(scale_sizes, sp_sc, 'b-s', linewidth=2, markersize=6)
ax9.fill_between(scale_sizes, 1, sp_sc, alpha=0.15, color='blue')
ax9.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='CPU baseline')
ax9.set_title('GPU Speedup vs Size', fontweight='bold')
ax9.set_xlabel('Number of Leaves')
ax9.set_ylabel('Speedup (×)')
ax9.legend()
ax9.grid(True, linestyle='--', alpha=0.5)
for x, y in zip(scale_sizes, sp_sc):
    ax9.annotate(f'{y:.1f}×', xy=(x,y), xytext=(0,8),
                 textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig('merkle_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ All 9 graphs saved as merkle_dashboard.png")

In [ ]:
def merkle_proof(input_data, leaf_index):
    leaves = [sha256_3stage(r.tobytes()) for r in input_data]
    level  = leaves[:]
    proof  = []
    idx    = leaf_index
    while len(level) > 1:
        if idx % 2 == 0:
            sib = level[idx+1] if idx+1 < len(level) else level[idx]
            proof.append(('right', sib.hex()))
        else:
            proof.append(('left', level[idx-1].hex()))
        nxt = []
        for i in range(0, len(level), 2):
            l = level[i]
            r = level[i+1] if i+1 < len(level) else level[i]
            nxt.append(hashlib.sha256(l+r).digest())
        level = nxt
        idx //= 2

    try:
        leaf_text = input_data[leaf_index].tobytes().decode('utf-8').replace('\n',' ')
    except:
        leaf_text = input_data[leaf_index].tobytes().decode('utf-8', errors='replace')

    print(f"\n{'='*65}")
    print(f"  MERKLE PROOF — Leaf #{leaf_index}")
    print(f"{'='*65}")
    print(f"  Text chunk  : '{leaf_text}'")
    print(f"  Leaf hash   : {sha256_3stage(input_data[leaf_index].tobytes()).hex()}")
    print(f"\n  Audit path:")
    for i, (side, h) in enumerate(proof):
        print(f"    Level {i}  : {side} sibling = {h[:32]}...")
    print(f"\n  Root        : {level[0].hex()}")
    print(f"  Proof depth : {len(proof)} levels")
    print(f"{'='*65}")

merkle_proof(input_data, leaf_index=0)